# **Unified Military Analytics and Comparison Dashboard**

**KPI Engineering & Feature Creation:**

**Objective:**
Transform cleaned military dataset into analytical KPIs to support:

1. Power Comparison Dashboard
2. Resource Efficiency Dashboard
3. Regional Analysis Dashboard
4. Alliance & Strategic Insights Dashboard

This step converts raw military statistics into decision-support metrics.

Load the cleaned dataset and inspect structure before KPI creation.

In [1]:
from google.colab import files
files.upload()

Saving military_cleaned_data.csv to military_cleaned_data.csv


{'military_cleaned_data.csv': b'country_name,power_index,available_manpower,purchasing_power_parity,defense_budget,active_personnel,reserve_personnel,air_force_personnel,army_personnel,navy_personnel,aircraft_total,attack,fighters,transport,helicopters,special_mission,trainers,attack_helicopters,tanker,naval_assets,aircraft_carriers,helicopter_carriers,destroyers,frigates,corvettes,submarines,patrol_vessels,mine_warfare,tanks,self_propelled_artillery,towed_artillery,rocket_artillery,external_debt\nAfghanistan,2.7342,15647405,82238000000,145000000,75000,0,0,165000,0,5,0,0,2,3,0,0,0,0,100,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,1929750000\nAlbania,1.7258,1522479,51360000000,720037190,7500,0,650,2335,1000,20,0,0,1,19,0,0,0,0,33,0.0,0.0,0.0,0.0,0.0,0.0,33.0,0.0,0,10,0,270,5363000000\nAlgeria,0.4849,22570787,722912000000,25000000000,130000,150000,14000,280000,15000,620,42,111,63,300,10,90,74,4,111,0.0,0.0,0.0,8.0,8.0,6.0,75.0,3.0,1485,224,483,188,4764000000\nAngola,1.1045,7440412,2782390000

In [2]:
import pandas as pd # Used for data manipulation and handling DataFrames
import numpy as np # Used for numerical operations and handling missing values (NaN)

# Load cleaned dataset
df = pd.read_csv("military_cleaned_data.csv")

# Preview first 5 rows
df.head()

# Dataset information
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 153 entries, 0 to 152
Data columns (total 33 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   country_name              153 non-null    object 
 1   power_index               153 non-null    float64
 2   available_manpower        153 non-null    int64  
 3   purchasing_power_parity   153 non-null    int64  
 4   defense_budget            153 non-null    int64  
 5   active_personnel          153 non-null    int64  
 6   reserve_personnel         153 non-null    int64  
 7   air_force_personnel       153 non-null    int64  
 8   army_personnel            153 non-null    int64  
 9   navy_personnel            153 non-null    int64  
 10  aircraft_total            153 non-null    int64  
 11  attack                    153 non-null    int64  
 12  fighters                  153 non-null    int64  
 13  transport                 153 non-null    int64  
 14  helicopter

**KPI Definitions:**

**Assets per Capita**

Measures military hardware relative to manpower.

**Formula:**
(total_aircraft + tanks + naval_assets) / available_manpower

**Budget-to-GDP Ratio**

Measures defense spending weight in national economy.

**Formula:**
defense_budget / purchasing_power_parity

**Power Index Rank Gap**

Measures distance from strongest country.

**Formula:**
rank(country) – rank(top_country)

**KPI Calculations:**

We compute KPIs while handling:

- Division by zero
- Missing values
- Ranking precision

In [3]:
# Ensure numeric columns
numeric_cols = [
    'aircraft_total', 'tanks', 'naval_assets',
    'available_manpower', 'defense_budget',
    'purchasing_power_parity', 'power_index'
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df.get(col), errors='coerce').fillna(0)

# ---- 1. Total Military Hardware ----
df['total_military_hardware'] = (
    df['aircraft_total'] +
    df['tanks'] +
    df['naval_assets']
)

# ---- 2. Assets per Capita ----
df['assets_per_capita'] = (
    df['total_military_hardware'] /
    df['available_manpower'].replace(0, np.nan)
).round(7)

# ---- 3. Budget-to-GDP Ratio ----
df['budget_to_gdp_ratio'] = (
    df['defense_budget'] /
    df['purchasing_power_parity'].replace(0, np.nan)
).round(5)

# ---- 4. Power Index Rank ----
df['power_index_rank'] = df['power_index'].rank(ascending=True).astype(int)

top_rank_value = df['power_index_rank'].min()

df['power_index_rank_gap'] = df['power_index_rank'] - top_rank_value

df.head()

,country_name,power_index,available_manpower,purchasing_power_parity,defense_budget,active_personnel,reserve_personnel,air_force_personnel,army_personnel,navy_personnel,...,tanks,self_propelled_artillery,towed_artillery,rocket_artillery,external_debt,total_military_hardware,assets_per_capita,budget_to_gdp_ratio,power_index_rank,power_index_rank_gap
0,Afghanistan,2.7342,15647405,82238000000,145000000,75000,0,0,165000,0,...,0,0,0,0,1929750000,105,0.000007,0.00176,125,124
1,Albania,1.7258,1522479,51360000000,720037190,7500,0,650,2335,1000,...,0,10,0,270,5363000000,53,0.000035,0.01402,75,74
2,Algeria,0.4849,22570787,722912000000,25000000000,130000,150000,14000,280000,15000,...,1485,224,483,188,4764000000,2216,0.000098,0.03458,26,25
3,Angola,1.1045,7440412,278239000000,31200000000,107000,0,6000,100000,1500,...,319,25,575,116,45299000000,629,0.000084,0.11213,57,56
4,Argentina,0.5983,20677529,1213000000000,993919790,108000,12370,14000,108000,18370,...,362,20,172,27,74362000000,645,0.000031,0.00082,31,30


**Alliance Classification**

We classify countries into:

- NATO
- Non-Aligned

In [4]:
# List of NATO member countries
nato_list = [
    "albania", "belgium", "bulgaria", "canada", "croatia", "czechia",
    "denmark", "estonia", "finland", "france", "germany", "greece",
    "hungary", "iceland", "italy", "latvia", "lithuania", "luxembourg",
    "montenegro", "netherlands", "north-macedonia", "norway", "poland",
    "portugal", "romania", "slovakia", "slovenia", "spain", "sweden",
    "turkiye", "united-kingdom", "united-states-of-america"
]

# Create a temporary standardized country key for matching
df['country_key'] = (
    df['country_name']
    .str.lower()
    .str.strip()
    .str.replace(' ', '-', regex=False)
)

# Create alliance flag column
df['alliance_flag'] = df['country_key'].apply(
    lambda x: 'NATO' if x in nato_list else 'Non-Aligned'
)

# Remove temporary column
df.drop(columns=['country_key'], inplace=True)

df[['country_name', 'alliance_flag']].head()

,country_name,alliance_flag
0,Afghanistan,Non-Aligned
1,Albania,NATO
2,Algeria,Non-Aligned
3,Angola,Non-Aligned
4,Argentina,Non-Aligned


Geographic Metadata Enrichment

- Continent
- Sub-region

In [5]:
import pandas as pd

In [6]:
def finalize_military_metadata(file_path):

    df = pd.read_csv(file_path)

    # ---- FIX 1: Standardize column names ----
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

    # Ensure country column exists
    if 'country_name' not in df.columns:
        raise ValueError("'country_name' column not found. Check column names.")

    def get_geo_details(country):
        c = str(country).lower().strip()

        # Americas
        if any(x in c for x in ['united-states', 'canada']):
            return 'Americas', 'Northern America'
        if any(x in c for x in ['chile', 'brazil', 'argentina', 'colombia', 'peru', 'ecuador', 'venezuela', 'uruguay', 'paraguay', 'bolivia', 'suriname']):
            return 'Americas', 'South America'
        if any(x in c for x in ['mexico', 'cuba', 'panama', 'guatemala', 'belize', 'dominican-republic', 'el-salvador', 'honduras', 'nicaragua']):
            return 'Americas', 'Central America'

        # Asia
        if any(x in c for x in ['india', 'pakistan', 'afghanistan', 'bangladesh', 'sri-lanka', 'nepal', 'bhutan']):
            return 'Asia', 'Southern Asia'
        if any(x in c for x in ['china', 'japan', 'korea', 'taiwan', 'mongolia']):
            return 'Asia', 'Eastern Asia'
        if any(x in c for x in ['vietnam', 'thailand', 'indonesia', 'philippines', 'singapore', 'malaysia', 'cambodia', 'laos', 'myanmar']):
            return 'Asia', 'South-East Asia'
        if any(x in c for x in ['kazakhstan', 'uzbekistan', 'turkmenistan', 'kyrgyzstan', 'tajikistan']):
            return 'Asia', 'Central Asia'

        # Europe
        if any(x in c for x in ['russia', 'ukraine', 'poland', 'romania', 'belarus', 'czechia', 'hungary', 'bulgaria', 'moldova', 'albania']):
            return 'Europe', 'Eastern Europe'
        if any(x in c for x in ['france', 'germany', 'belgium', 'netherlands', 'switzerland', 'austria', 'luxembourg']):
            return 'Europe', 'Western Europe'
        if any(x in c for x in ['united-kingdom', 'sweden', 'norway', 'denmark', 'finland', 'ireland', 'iceland', 'estonia', 'latvia', 'lithuania']):
            return 'Europe', 'Northern Europe'
        if any(x in c for x in ['italy', 'spain', 'greece', 'portugal', 'serbia', 'croatia', 'slovenia', 'bosnia', 'montenegro', 'north-macedonia', 'kosovo']):
            return 'Europe', 'Southern Europe'

        # Middle East
        if any(x in c for x in ['israel', 'saudi', 'turkiye', 'iran', 'iraq', 'uae', 'qatar', 'jordan', 'kuwait', 'lebanon', 'oman', 'yemen', 'bahrain', 'syria', 'cyprus', 'georgia', 'armenia', 'azerbaijan']):
            return 'Middle East', 'Western Asia (Middle East)'

        # Africa
        if any(x in c for x in ['egypt', 'algeria', 'morocco', 'libya', 'tunisia']):
            return 'Africa', 'Northern Africa'
        if any(x in c for x in ['nigeria', 'south-africa', 'kenya', 'ethiopia', 'angola', 'benin', 'botswana', 'burundi', 'cameroon', 'central-african-republic', 'chad', 'congo', 'djibouti', 'eritrea', 'gabon', 'gambia', 'ghana', 'guinea', 'ivory-coast', 'lesotho', 'liberia', 'madagascar', 'malawi', 'mali', 'mauritania', 'mauritius', 'mozambique', 'namibia', 'niger', 'senegal', 'sierra-leone', 'somalia', 'sudan', 'tanzania', 'uganda', 'zambia', 'zimbabwe']):
            return 'Africa', 'Sub-Saharan Africa'

        # Oceania
        if any(x in c for x in ['australia', 'new-zealand']):
            return 'Oceania', 'Australia & New Zealand'

        return 'Other', 'Other Subregion'

    df[['continent', 'region']] = df['country_name'].apply(
        lambda x: pd.Series(get_geo_details(x))
    )

    df.to_csv('military_final_v2.csv', index=False)

    print("Success: Geographic metadata added successfully.")
    return df


# Run function
finalize_military_metadata("military_cleaned_data.csv")

Success: Geographic metadata added successfully.


,country_name,power_index,available_manpower,purchasing_power_parity,defense_budget,active_personnel,reserve_personnel,air_force_personnel,army_personnel,navy_personnel,...,submarines,patrol_vessels,mine_warfare,tanks,self_propelled_artillery,towed_artillery,rocket_artillery,external_debt,continent,region
0,Afghanistan,2.7342,15647405,82238000000,145000000,75000,0,0,165000,0,...,0.0,0.0,0.0,0,0,0,0,1929750000,Asia,Southern Asia
1,Albania,1.7258,1522479,51360000000,720037190,7500,0,650,2335,1000,...,0.0,33.0,0.0,0,10,0,270,5363000000,Europe,Eastern Europe
2,Algeria,0.4849,22570787,722912000000,25000000000,130000,150000,14000,280000,15000,...,6.0,75.0,3.0,1485,224,483,188,4764000000,Africa,Northern Africa
3,Angola,1.1045,7440412,278239000000,31200000000,107000,0,6000,100000,1500,...,0.0,32.0,0.0,319,25,575,116,45299000000,Africa,Sub-Saharan Africa
4,Argentina,0.5983,20677529,1213000000000,993919790,108000,12370,14000,108000,18370,...,2.0,13.0,0.0,362,20,172,27,74362000000,Americas,South America
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148,Venezuela,0.9106,15625153,110943000000,6060000000,109000,8000,10755,300745,25500,...,3.0,26.0,0.0,282,60,92,56,121363000000,Americas,South America
149,Vietnam,0.4066,54994667,1456000000000,10200000000,450000,5000000,35000,5450000,50000,...,6.0,27.0,8.0,1999,30,3250,774,34426000000,Asia,South-East Asia
150,Yemen,2.1617,12213368,18719000000,405187500,33350,0,3850,56350,3250,...,0.0,9.0,0.0,0,12,60,30,6492000000,Middle East,Western Asia (Middle East)
151,Zambia,2.4461,7279691,79207000000,385321000,15150,0,3745,13250,0,...,0.0,0.0,0.0,75,12,60,50,16597000000,Africa,Sub-Saharan Africa


**KPI Validation**

We validate:

1. No division by zero errors
2. No infinite values
3. No unrealistic KPI spikes

In [7]:
# Ensure numeric columns, as df might have been reloaded or reset
numeric_cols = [
    'aircraft_total', 'tanks', 'naval_assets',
    'available_manpower', 'defense_budget',
    'purchasing_power_parity', 'power_index'
]

for col in numeric_cols:
    # Use .get() for robustness in case column is entirely missing
    df[col] = pd.to_numeric(df.get(col, 0), errors='coerce').fillna(0)

# Recalculate 'total_military_hardware'
df['total_military_hardware'] = (
    df['aircraft_total'] +
    df['tanks'] +
    df['naval_assets']
)

# Recalculate 'assets_per_capita'
df['assets_per_capita'] = (
    df['total_military_hardware'] /
    df['available_manpower'].replace(0, np.nan)
).round(7)

# Recalculate 'budget_to_gdp_ratio'
df['budget_to_gdp_ratio'] = (
    df['defense_budget'] /
    df['purchasing_power_parity'].replace(0, np.nan)
).round(5)

# Recalculate 'power_index_rank' and 'power_index_rank_gap'
df['power_index_rank'] = df['power_index'].rank(ascending=True).astype(int)
top_rank_value = df['power_index_rank'].min()
df['power_index_rank_gap'] = df['power_index_rank'] - top_rank_value

# Recalculate 'alliance_flag'
nato_list = [
    "albania", "belgium", "bulgaria", "canada", "croatia", "czechia",
    "denmark", "estonia", "finland", "france", "germany", "greece",
    "hungary", "iceland", "italy", "latvia", "lithuania", "luxembourg",
    "montenegro", "netherlands", "north-macedonia", "norway", "poland",
    "portugal", "romania", "slovakia", "slovenia", "spain", "sweden",
    "turkiye", "united-kingdom", "united-states-of-america"
]

# Create a temporary standardized country key for matching
df['country_key'] = (
    df['country_name']
    .str.lower()
    .str.strip()
    .str.replace(' ', '-', regex=False)
)

df['alliance_flag'] = df['country_key'].apply(
    lambda x: 'NATO' if x in nato_list else 'Non-Aligned'
)

df.drop(columns=['country_key'], inplace=True)

# After ensuring all necessary columns are present, proceed with original validation
# Generate descriptive statistics
df.describe()

# Check missing values in each column
df.isna().sum()

# Identify countries with highest assets per capita
df[df['assets_per_capita'] > 0.01].sort_values(
    by='assets_per_capita', ascending=False
).head()

,country_name,power_index,available_manpower,purchasing_power_parity,defense_budget,active_personnel,reserve_personnel,air_force_personnel,army_personnel,navy_personnel,...,self_propelled_artillery,towed_artillery,rocket_artillery,external_debt,total_military_hardware,assets_per_capita,budget_to_gdp_ratio,power_index_rank,power_index_rank_gap,alliance_flag


Export final analytical dataset for dashboard layer.

In [8]:
df.to_csv("military_final.csv", index=False)

print("Final dataset exported successfully.")

Final dataset exported successfully.


This completes the KPI Engineering phase of the project.